# ADTC 2026 — Fine-tune & quantize on Colab (T4)

Runs the whole model pipeline on a Colab GPU, which solves the two things a
low-spec laptop can't: a real GPU and a fast network.

**Pipeline:** clone repo -> LoRA fine-tune Qwen3.5-0.8B on the IMCI corpus ->
merge -> convert to GGUF (`--no-mtp`) -> imatrix + Q4_K_M with the base's
per-tensor profile -> score fine-tuned vs base -> publish the `.gguf` to HF
(which is also what `download_model.sh` fetches).

**Before you start:**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Push the R&D repo (with `data/sft/` and `scripts/`) to GitHub so this can clone it.
3. Have a Hugging Face **write token** ready (Settings -> Access Tokens) for the publish step.

The whole run is ~30-45 min on a free T4 — well within session limits. Save the
outputs (publish to HF) before the session disconnects.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Install the training stack
`transformers>=5` is required for the `qwen3_5` (hybrid Mamba/attention) class —
Colab ships an older one. **If Colab asks to restart the runtime after this, do it,
then continue from cell 3** (don't rerun this cell).

In [ ]:
!pip install -q -U "transformers>=5" peft trl accelerate "huggingface_hub[cli]"
# Colab preinstalls an old torchao (0.10) that peft's LoRA dispatch rejects on a
# version check — even though we don't use torchao (plain fp16). Remove it so
# peft cleanly skips that path.
!pip uninstall -y -q torchao 2>/dev/null; echo "torchao removed (if it was present)"
import transformers, peft, trl; print('transformers', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__)
from transformers.models import qwen3_5; print('qwen3_5 modeling class present:', qwen3_5.__name__)

## 3. Clone the R&D repo (scripts + IMCI corpus)
Edit the URL if your fork lives elsewhere. The base model weights are **not** in the
repo (gitignored) — they're pulled from HF in the training cell.

In [ ]:
%cd /content
![ -d adtc-agent ] || git clone https://github.com/Jayayjay/adtc-agent.git
%cd /content/adtc-agent
!ls data/sft/*.jsonl && wc -l data/sft/train.jsonl

## 4. LoRA fine-tune (the main event)
`--fp16` is native on T4. `--max-seq-len 512` is ample (corpus max is 314 tokens).
Base `Qwen/Qwen3.5-0.8B` is apache-2.0 and ungated; it downloads on first use (~1.7 GB).

If the SSM/Mamba layers error on this GPU (kernel availability), re-run with
`--freeze-ssm` added — it LoRA-adapts attention + FFN only. (On CPU the gradients
flow fine, so this is a low-probability escape hatch.)

In [ ]:
!python scripts/train_lora.py \
    --model Qwen/Qwen3.5-0.8B \
    --data-dir data/sft \
    --output-dir /content/lora_adapter \
    --epochs 2 --lora-r 16 --lr 1e-4 \
    --batch-size 4 --grad-accum 4 --max-seq-len 512 \
    --fp16 --logging-steps 20
!ls -la /content/lora_adapter

## 5. Build llama.cpp (converter + quantizer)
Needed for GGUF conversion, imatrix, and quantization. ~5 min.

In [ ]:
%cd /content
![ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF >/dev/null 2>&1
!cmake --build llama.cpp/build --target llama-quantize llama-imatrix -j4 >/dev/null 2>&1
!ls llama.cpp/build/bin/llama-quantize llama.cpp/build/bin/llama-imatrix

## 6. Merge -> GGUF -> imatrix -> Q4_K_M
Replicates the base model's per-tensor SSM quant profile (a plain Q4_K_M is
*smaller* because it's *worse*). The calib file mixes IMCI + general text.

In [ ]:
%cd /content/adtc-agent
!python scripts/merge_and_quantize.py \
    --adapter /content/lora_adapter \
    --base Qwen/Qwen3.5-0.8B \
    --llama-cpp /content/llama.cpp \
    --calib data/sft/imatrix_calib.txt \
    --out /content/Qwen3.5-0.8B-IMCI-Q4_K_M.gguf
!ls -lh /content/Qwen3.5-0.8B-IMCI-Q4_K_M.gguf

## 7. Score it (the before/after the report needs)
Runs the fine-tuned `.gguf` through the natural-language vignettes and reports
accuracy + **under-triage rate** at greedy and temp 0.8. Uses `llama-cpp-python`.

For a base A/B, download the base GGUF and pass `--base` (optional).

In [ ]:
!pip install -q llama-cpp-python
!python eval/scoring/model_sacc_scorer.py \
    --model /content/Qwen3.5-0.8B-IMCI-Q4_K_M.gguf \
    --tasks eval/tasks/imci_vignettes_nl.json
!cat eval/results/model_sacc.json

## 8. Publish the GGUF to Hugging Face
This persists the model past the session **and** provides the public URL that
`download_model.sh` fetches. Paste a **write** token when prompted.

In [ ]:
from huggingface_hub import login, HfApi
login()  # paste a WRITE token
repo = 'Jayayjay/Qwen3.5-0.8B-IMCI-GGUF'  # <- your public repo id
api = HfApi()
api.create_repo(repo, repo_type='model', exist_ok=True, private=False)
api.upload_file(
    path_or_fileobj='/content/Qwen3.5-0.8B-IMCI-Q4_K_M.gguf',
    path_in_repo='Qwen3.5-0.8B-IMCI-Q4_K_M.gguf',
    repo_id=repo, repo_type='model')
print(f'Published. download_model.sh URL:\n  https://huggingface.co/{repo}/resolve/main/Qwen3.5-0.8B-IMCI-Q4_K_M.gguf')

## 9. (Backup) save the adapter to Google Drive
Optional — the published GGUF is the deliverable, but keeping the adapter lets you
re-quantize later without retraining.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/adtc && cp -r /content/lora_adapter /content/drive/MyDrive/adtc/ && cp /content/Qwen3.5-0.8B-IMCI-Q4_K_M.gguf /content/drive/MyDrive/adtc/
!ls -la /content/drive/MyDrive/adtc/